# LearnMateAI — Qwen 2.5 LoRA Fine-Tuning

Colab-ready notebook for **LoRA/PEFT** fine-tuning (not full fine-tuning) on Stage 3 dataset output.

## Golden path (avoids every error hit while building this)

1. **Runtime → Disconnect and delete runtime** (start clean if you tried installs before)
2. **Runtime → Change runtime type → GPU → T4**
3. Run cell **"0 — Install dependencies"** once
4. **Runtime → Restart session** (required — do not skip)
5. Run cell **"0b — Verify environment"** — every line should print a version, not `IMPORT FAILED`
6. Run the rest top to bottom: CONFIG → dataset → tokenizer/model → LoRA → train → save → run-record → download

**Never** run `pip install --force-reinstall` on `torch`, `torchvision`, `pillow`, or `numpy` in Colab — that is what caused the `torchvision::nms`, `PIL._typing._Ink`, and `numpy.dtype size changed` errors during development. Colab's preinstalled build of those four packages already matches its GPU driver/CUDA; leave them alone and only add the Hugging Face training libraries.

**Budget note (~USD 45/mo ops):** Prefer `Qwen/Qwen2.5-1.5B-Instruct` + QLoRA on free/cheap Colab. Larger bases only if a sponsored GPU is available.

**Status:** Fine-tuning logic in this notebook is complete and hardened against the dependency issues hit on live Colab runtimes. A full GPU training run has **not** yet been executed and recorded in this repository (see top-level `model-Thevindu/README.md`) — run it, then commit the resulting `run_record.json`. The dataset paths below default to `sample_data/` (`lm-legal-smoke-v1`, a synthetic smoke set) — swap in a real Stage 3 dataset before treating a run as production-worthy.

## 0 — Install dependencies (Colab)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # SAFE INSTALL for Colab.
    # We deliberately do NOT touch torch / torchvision / pillow / numpy -- Colab's
    # preinstalled versions of those four are already matched to its GPU driver and
    # CUDA build. Reinstalling or upgrading them is what caused, in order, while
    # building this notebook:
    #   ValueError: numpy.dtype size changed, may indicate binary incompatibility
    #   RuntimeError: operator torchvision::nms does not exist
    #   ImportError: cannot import name '_Ink' from 'PIL._typing'
    # Only the Hugging Face training stack is installed/upgraded here.
    %pip install -q -U \
        transformers \
        accelerate \
        peft \
        bitsandbytes \
        trl \
        datasets \
        sentencepiece \
        einops
    print("Installed Hugging Face training stack (transformers/accelerate/peft/bitsandbytes/trl/datasets).")
    print("NEXT STEP (required): Runtime -> Restart session, then run cell '0b - Verify environment'.")
    print("Do not re-run this install cell in the same session.")
else:
    print("Not running in Colab -- install matching versions from requirements.txt locally.")

print("IN_COLAB =", IN_COLAB)

## 0b — Verify environment (run AFTER "Runtime → Restart session")

Do **not** re-run the install cell above after this. If any line below prints `IMPORT FAILED`, see the troubleshooting note printed at the bottom of this cell's output.

In [ ]:
import importlib


def _v(mod: str) -> str:
    try:
        m = importlib.import_module(mod)
        return getattr(m, "__version__", "unknown")
    except Exception as e:  # noqa: BLE001
        return f"IMPORT FAILED: {type(e).__name__}: {e}"


import torch

print("torch          :", torch.__version__)
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            :", torch.cuda.get_device_name(0))
    print("capability     :", torch.cuda.get_device_capability(0))

for _mod in ["transformers", "accelerate", "peft", "bitsandbytes", "trl", "datasets"]:
    print(f"{_mod:14s}:", _v(_mod))

print(
    "\nIf anything above says IMPORT FAILED: Runtime -> Disconnect and delete runtime, "
    "reconnect on a FRESH GPU runtime, run ONLY the install cell once, Restart session, "
    "then re-run this cell. Do not run the install cell twice in the same runtime."
)

## 1 — CONFIG (single source of truth)

Change hyperparameters **only here**. The run-record cell reads this dict so every saved adapter is reproducible.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

CONFIG = {
    # --- Identity ---
    "run_id": f"qwen25-lora-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
    "project": "LearnMateAI",
    "track": "model-Thevindu",

    # --- Base model ---
    "base_model_id": "Qwen/Qwen2.5-1.5B-Instruct",  # upgrade to 3B/7B only with GPU budget
    "torch_dtype": "bfloat16",  # fallback to float16 on older GPUs
    "use_qlora": True,          # 4-bit; set False for full LoRA in bf16 if VRAM allows

    # --- Dataset (Stage 3 output) ---
    "dataset_version": "lm-legal-smoke-v1",
    "train_path": "sample_data/train.jsonl",  # Colab: upload or mount Drive path
    "val_path": "sample_data/val.jsonl",
    "max_seq_length": 1024,
    "packing": False,

    # --- LoRA / PEFT ---
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "bias": "none",
        "task_type": "CAUSAL_LM",
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    },

    # --- Training ---
    "training": {
        "num_train_epochs": 2,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 5,
        "eval_strategy": "steps",
        "eval_steps": 25,
        "save_strategy": "steps",
        "save_steps": 25,
        "save_total_limit": 2,
        "fp16": False,
        "bf16": True,
        "optim": "paged_adamw_8bit",
        "report_to": "none",
        "seed": 42,
    },

    # --- Outputs ---
    "output_dir": "adapters",
    "run_records_dir": "run_records",
}

ADAPTER_DIR = Path(CONFIG["output_dir"]) / CONFIG["run_id"]
RUN_RECORD_PATH = Path(CONFIG["run_records_dir"]) / f"{CONFIG['run_id']}.json"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
Path(CONFIG["run_records_dir"]).mkdir(parents=True, exist_ok=True)

print("run_id       :", CONFIG["run_id"])
print("base_model   :", CONFIG["base_model_id"])
print("dataset      :", CONFIG["dataset_version"])
print("adapter_dir  :", ADAPTER_DIR)
print("run_record   :", RUN_RECORD_PATH)

## 2 — Load Stage 3 JSONL and format for chat SFT

In [ ]:
import json
from pathlib import Path
from datasets import Dataset


def _ensure_dataset_files() -> None:
    """If train/val JSONL are missing, offer a Colab upload widget instead of failing."""
    train_p = Path(CONFIG["train_path"])
    val_p = Path(CONFIG["val_path"])
    if train_p.exists() and val_p.exists():
        return

    print("Dataset files not found:")
    print(" ", train_p, "exists=", train_p.exists())
    print(" ", val_p, "exists=", val_p.exists())

    if IN_COLAB:
        print(
            "\nUpload dialog opening -- select train.jsonl, val.jsonl "
            "(and test.jsonl if you have it) from model-Thevindu/02_finetuning/sample_data/."
        )
        train_p.parent.mkdir(parents=True, exist_ok=True)
        from google.colab import files

        uploaded = files.upload()
        for name, content in uploaded.items():
            dest = train_p.parent / name
            with open(dest, "wb") as f:
                f.write(content)
            print("saved", dest)

        if not (train_p.exists() and val_p.exists()):
            raise FileNotFoundError(
                f"Still missing after upload: {train_p} and/or {val_p}. "
                "Filenames must match CONFIG['train_path'] / CONFIG['val_path']."
            )
    else:
        raise FileNotFoundError(
            f"Missing dataset files: {train_p} / {val_p}. "
            "Copy model-Thevindu/02_finetuning/sample_data/ next to this notebook."
        )


_ensure_dataset_files()


def load_jsonl(path: str):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(CONFIG["train_path"])
val_rows = load_jsonl(CONFIG["val_path"])

assert train_rows, f"Empty train set: {CONFIG['train_path']}"
assert all("messages" in r for r in train_rows), "Expected chat `messages` field from Stage 3"

# Confirm dataset_version lineage
versions = {r.get("dataset_version") for r in train_rows}
print(f"train={len(train_rows)}  val={len(val_rows)}  dataset_versions={versions}")
if CONFIG["dataset_version"] not in versions:
    print("WARNING: CONFIG dataset_version does not match records — update CONFIG before a real run.")

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
train_ds[0]["messages"][:2]

## 3 — Tokenizer + base model (QLoRA or LoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this notebook. Runtime -> Change runtime type -> GPU (T4).")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model_id"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4 (compute capability 7.5) has no real bf16 tensor cores -- use fp16 there.
# Newer GPUs (A100/L4/H100, capability >= 8) use bf16.
_cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if _cap_major >= 8 else torch.float16
CONFIG["torch_dtype_actual"] = str(dtype)
print(f"GPU capability {torch.cuda.get_device_capability(0)} -> using dtype {dtype}")


def _load_model(use_qlora: bool):
    quant_cfg = None
    if use_qlora:
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
    return AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model_id"],
        quantization_config=quant_cfg,
        device_map="auto",
        # Always pass torch_dtype explicitly (never None). With QLoRA, this only
        # governs the non-quantized modules -- but leaving it None makes
        # from_pretrained fall back to the checkpoint's own declared dtype
        # (Qwen ships as bfloat16), which then leaks into the LoRA adapter layers
        # built on top. That desyncs them from an fp16 GradScaler on GPUs without
        # bf16 support (e.g. T4) and raises:
        #   NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'
        torch_dtype=dtype,
        trust_remote_code=True,
    )


try:
    model = _load_model(CONFIG["use_qlora"])
except Exception as exc:  # noqa: BLE001 -- fall back rather than hard-fail the whole run
    print(f"QLoRA (4-bit) load failed: {type(exc).__name__}: {exc}")
    print("Falling back to non-quantized LoRA load (uses more VRAM).")
    CONFIG["use_qlora"] = False
    model = _load_model(False)

model.config.use_cache = False
print("loaded", CONFIG["base_model_id"], "qlora=" + str(CONFIG["use_qlora"]))

## 4 — Attach LoRA adapters (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if CONFIG["use_qlora"]:
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(**CONFIG["lora"])
model = get_peft_model(model, lora_cfg)

# Force every trainable (LoRA) param to a SAFE dtype for the optimizer/scaler:
#   - bf16 GPUs (Ampere+): trainable params can stay in bfloat16 directly (no
#     GradScaler is used for bf16, so there's no dtype restriction).
#   - fp16 GPUs (e.g. T4): trainable params MUST be float32, not float16.
#     PyTorch's GradScaler (used for fp16 AMP) explicitly refuses to unscale
#     native float16 gradients:  ValueError: "Attempting to unscale FP16
#     gradients." The fp16 speed-up still comes from autocast on the forward/
#     backward math; the actual params/grads the optimizer touches must stay fp32.
# (Leaving adapter weights in whatever the checkpoint happened to declare has
# also caused: NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'.)
_lora_param_dtype = torch.bfloat16 if dtype == torch.bfloat16 else torch.float32
_n_recast = 0
for _name, _param in model.named_parameters():
    if _param.requires_grad and _param.dtype != _lora_param_dtype:
        _param.data = _param.data.to(_lora_param_dtype)
        _n_recast += 1
if _n_recast:
    print(f"Recast {_n_recast} trainable tensor(s) to {_lora_param_dtype}.")

model.print_trainable_parameters()

## 5 — Train with TRL SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
    _HAS_SFTCONFIG = True
except ImportError:
    _HAS_SFTCONFIG = False


def formatting_func(example):
    # Qwen chat template applied by tokenizer
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )


t = CONFIG["training"]
# Older GPUs may not support bf16
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
fp16 = not bf16_ok
bf16 = bf16_ok and t["bf16"]

base_args = dict(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=t["num_train_epochs"],
    per_device_train_batch_size=t["per_device_train_batch_size"],
    per_device_eval_batch_size=t["per_device_eval_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    learning_rate=t["learning_rate"],
    lr_scheduler_type=t["lr_scheduler_type"],
    warmup_ratio=t["warmup_ratio"],
    weight_decay=t["weight_decay"],
    logging_steps=t["logging_steps"],
    save_strategy=t["save_strategy"],
    save_steps=t["save_steps"],
    save_total_limit=t["save_total_limit"],
    fp16=fp16,
    bf16=bf16,
    optim=t["optim"],
    report_to=t["report_to"],
    seed=t["seed"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# transformers renamed evaluation_strategy -> eval_strategy at one point; try both.
training_args = None
for _eval_kw in (
    {"eval_strategy": t["eval_strategy"], "eval_steps": t["eval_steps"]},
    {"evaluation_strategy": t["eval_strategy"], "eval_steps": t["eval_steps"]},
):
    try:
        training_args = TrainingArguments(**base_args, **_eval_kw)
        break
    except TypeError:
        continue
if training_args is None:
    print("WARNING: could not set an eval-strategy kwarg on TrainingArguments; using defaults.")
    training_args = TrainingArguments(**base_args)

# SFTTrainer's constructor signature has changed across trl releases (tokenizer= vs
# processing_class=, with/without max_seq_length, TrainingArguments vs SFTConfig).
# Try the combinations in order instead of hardcoding one that might not match
# whatever trl version Colab installed.
trainer = None
_last_err = None
_attempts = [
    {"args": training_args, "tok_kw": "tokenizer", "use_msl": True},
    {"args": training_args, "tok_kw": "processing_class", "use_msl": True},
    {"args": training_args, "tok_kw": "processing_class", "use_msl": False},
    {"args": training_args, "tok_kw": "tokenizer", "use_msl": False},
]

for i, a in enumerate(_attempts, start=1):
    try:
        kwargs = dict(
            model=model,
            args=a["args"],
            train_dataset=train_ds,
            eval_dataset=val_ds,
            formatting_func=formatting_func,
        )
        if a["use_msl"]:
            kwargs["max_seq_length"] = CONFIG["max_seq_length"]
        kwargs[a["tok_kw"]] = tokenizer
        trainer = SFTTrainer(**kwargs)
        print(f"SFTTrainer constructed (attempt {i}: {a['tok_kw']}, max_seq_length={a['use_msl']})")
        break
    except TypeError as e:
        _last_err = e

if trainer is None and _HAS_SFTCONFIG:
    print("Falling back to SFTConfig-based construction (newer trl API)...")
    sft_kwargs = dict(base_args)
    sft_kwargs["max_seq_length"] = CONFIG["max_seq_length"]
    sft_kwargs["packing"] = CONFIG["packing"]
    try:
        sft_args = SFTConfig(**sft_kwargs)
    except TypeError:
        sft_kwargs.pop("max_seq_length", None)
        sft_kwargs.pop("packing", None)
        sft_args = SFTConfig(**sft_kwargs)
    for tok_kw in ("processing_class", "tokenizer"):
        try:
            trainer = SFTTrainer(
                model=model,
                args=sft_args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                formatting_func=formatting_func,
                **{tok_kw: tokenizer},
            )
            print(f"SFTTrainer constructed via SFTConfig ({tok_kw})")
            break
        except TypeError as e:
            _last_err = e

if trainer is None:
    raise RuntimeError(f"Could not construct SFTTrainer with the installed trl version. Last error: {_last_err}")

# Final precision guard, independent of the model-loading/LoRA cells above.
# If this cell is re-run without re-running cells 3-4 first (e.g. after a fix was
# pulled in but the kernel wasn't restarted), `model` may still hold trainable
# (LoRA) params in a dtype that disagrees with the fp16/bf16 decision made just
# above. Re-check and force-fix it right before training starts, using THIS
# cell's own fp16/bf16 flags as the source of truth.
#
# IMPORTANT: trainable params must NEVER be stored as native float16 here.
# PyTorch's GradScaler (used whenever fp16=True) explicitly rejects float16
# gradients: ValueError: "Attempting to unscale FP16 gradients." -- fp16 speed
# comes from autocast on the forward/backward math; the optimizer-visible
# params/grads must stay float32 under fp16 training. Only bf16 training (no
# scaler involved) can keep trainable params natively in bfloat16.
_train_dtype = torch.bfloat16 if bf16 else torch.float32
_n_recast = 0
for _name, _param in model.named_parameters():
    if _param.requires_grad and _param.dtype != _train_dtype:
        _param.data = _param.data.to(_train_dtype)
        _n_recast += 1
if _n_recast:
    print(f"[precision guard] Recast {_n_recast} trainable tensor(s) to {_train_dtype} (fp16={fp16}, bf16={bf16}).")
else:
    print(f"[precision guard] All trainable tensors already match {_train_dtype}.")

train_result = trainer.train()
FINAL_TRAIN_LOSS = float(train_result.training_loss)
print("final train loss:", FINAL_TRAIN_LOSS)

eval_metrics = trainer.evaluate()
FINAL_EVAL_LOSS = float(eval_metrics.get("eval_loss", float("nan")))
print("final eval loss:", FINAL_EVAL_LOSS)

## 6 — Save adapter weights

In [ ]:
adapter_save_path = ADAPTER_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_save_path))
tokenizer.save_pretrained(str(adapter_save_path))
print("adapter saved to", adapter_save_path)

## 7 — MANDATORY RUN RECORD

Writes hyperparameters, dataset version, and final loss next to the adapter.   An adapter without a run-record is not eligible for evaluation/promotion.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

# Fail loudly if training metrics were never produced
assert "FINAL_TRAIN_LOSS" in dir() or "FINAL_TRAIN_LOSS" in globals(), (
    "FINAL_TRAIN_LOSS missing — run the training cell before writing a run-record."
)

run_record = {
    "run_id": CONFIG["run_id"],
    "project": CONFIG["project"],
    "track": CONFIG["track"],
    "completed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "base_model_id": CONFIG["base_model_id"],
    "dataset_version": CONFIG["dataset_version"],
    "train_path": CONFIG["train_path"],
    "val_path": CONFIG["val_path"],
    "train_examples": len(train_rows),
    "val_examples": len(val_rows),
    "use_qlora": CONFIG["use_qlora"],
    "torch_dtype_actual": CONFIG.get("torch_dtype_actual", CONFIG["torch_dtype"]),
    "lora": CONFIG["lora"],
    "training": CONFIG["training"],
    "max_seq_length": CONFIG["max_seq_length"],
    "final_train_loss": FINAL_TRAIN_LOSS,
    "final_eval_loss": FINAL_EVAL_LOSS if "FINAL_EVAL_LOSS" in dir() or "FINAL_EVAL_LOSS" in globals() else None,
    "adapter_path": str(adapter_save_path),
    "status": "completed",
    "notes": "LoRA/PEFT only — base weights not modified. App must keep Gemini (or other API) fallback if adapter unavailable.",
}

# Persist beside adapter AND in run_records/
sidecar = Path(adapter_save_path) / "run_record.json"
for path in (RUN_RECORD_PATH, sidecar):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(run_record, f, indent=2)
    print("wrote", path)

run_record

## Next steps

1. Run cell **"8 — Download the adapter"** below to get `adapters/<run_id>/` off Colab (includes `adapter/` + `run_record.json`).
2. Log cost/duration in `04_docs/training_run_log.md`.
3. Evaluate with `03_testing_and_versioning/evaluate_candidate.ipynb` — do **not** promote without passing acceptance thresholds.

## 8 — Download the adapter (Colab)

Zips `adapters/<run_id>/` (LoRA weights + `run_record.json`) and triggers a browser download. Run this after training and the run-record cell above.

In [ ]:
import shutil
import sys

# 1. Define IN_COLAB to fix the NameError
IN_COLAB = 'google.colab' in sys.modules

# (Assuming ADAPTER_DIR is already defined in a previous cell)
zip_base = str(ADAPTER_DIR) 
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(ADAPTER_DIR))
print("Created:", zip_path)

# 2. Trigger the download
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    print("Browser download triggered. If nothing happens, check your browser's pop-up blocker.")
else:
    print(f"Not in Colab - find the zip at: {zip_path}")